# 16-phase16b-net2net-grow

**neuron Phase 16b** — paradigm 의 dynamic phase 두 번째 갈래. Phase 16a 가 *within-shape DST* (sparsity 재할당, parameter 수 동일) 였다면, Phase 16b 는 **cross-shape expansion** — 학습 중 `ffn_dim` 을 확장하여 *parameter 수 자체 증가*. **function preservation** 보장 (Net2Net-style: 새 input weight = 0).

핵심 가설:
1. **function preservation** — grow 직후 forward output 이 grow 전과 정확히 동일?
2. **grown ≤ large-baseline + threshold** — 작게 시작 + 중간에 grow 한 모델이 처음부터 큰 모델 근방까지 학습?
3. **grown < small-baseline** — grow 가 capacity 부족 모델보다 명확히 좋음?
4. **all-finite** — grow 후 학습 안정성?

설계: 3 mode × 2 seed = 6 run.
- `small_baseline`: ffn=128 끝까지 (small capacity)
- `grown`: ffn=128 시작 → step 750 에서 256 으로 grow
- `large_baseline`: ffn=256 끝까지 (large capacity)

arch 고정: `hybrid_around_one_around_one` + `use_full_graph=True` (Phase 14 최저 loss 구조)
데이터: TinyShakespeare (char-LM, block_size=64)
시드: [42, 123]
작성일: 2026-05-27
연관: Issue [#77](https://github.com/EinSofINTEREST/GraphLM/issues/77) (Phase 16 main [#75](https://github.com/EinSofINTEREST/GraphLM/issues/75)) / Phase 16a PR [#78](https://github.com/EinSofINTEREST/GraphLM/pull/78)

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridTransformerTrainConfig,
    train_hybrid_transformer_lm,
)
from graphlm.utils import safe_perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

## 1. Config + 데이터

In [ ]:
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

HIDDEN_DIM = 128
N_HEADS = 4
N_LAYERS = 4
GROUP_SIZE = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500
GROW_AT_STEP = MAX_STEPS // 2  # 750
SEEDS = [42, 123]
ARCH = "hybrid_around_one_around_one"

FFN_SMALL = 128
FFN_LARGE = 256

# 3 mode 정의
MODES = [
    {
        "name": "small_baseline",
        "ffn_dim": FFN_SMALL,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "grown",
        "ffn_dim": FFN_SMALL,
        "grow_at_step": GROW_AT_STEP,
        "grow_ffn_target": FFN_LARGE,
    },
    {
        "name": "large_baseline",
        "ffn_dim": FFN_LARGE,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
]
print(f"\nGrow 시점: step {GROW_AT_STEP} / {MAX_STEPS}, ffn_dim {FFN_SMALL} → {FFN_LARGE}")

## 2. Sweep 실행 (3 mode × 2 seed = 6 run)

In [ ]:
results = {}
for mode in MODES:
    for seed in SEEDS:
        key = (mode["name"], seed)
        print(f"\n== mode={mode['name']} seed={seed} ==")
        cfg = HybridTransformerTrainConfig(
            dataset=dataset,
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=mode["ffn_dim"],
            n_layers=N_LAYERS,
            group_size=GROUP_SIZE,
            arch=ARCH,
            use_full_graph=True,
            block_size=BLOCK_SIZE,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            grow_at_step=mode["grow_at_step"],
            grow_ffn_target=mode["grow_ffn_target"],
            seed=seed,
            device=device,
        )
        out = train_hybrid_transformer_lm(cfg)
        results[key] = out
        ev = out["grow_event"]
        print(
            f"  final_loss = {out['final_loss']:.4f}  (ppl = {safe_perplexity(out['final_loss']):.2f})"
            f"  params = {out['final_param_count']:,}"
            f"  grow_event = {ev}"
        )

## 3. 결과 표 + 자동 verdict

In [ ]:
print(f"{'mode':>16s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s} {'params':>10s}")
print("-" * 65)
for (name, seed), out in results.items():
    fl = out["final_loss"]
    print(
        f"{name:>16s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f} "
        f"{out['final_param_count']:>10,}"
    )

# mode 별 평균
print("\n== Mode summary (mean ± σ across seeds) ==")
summary = {}
for mode in MODES:
    name = mode["name"]
    vals = [results[(name, s)]["final_loss"] for s in SEEDS]
    params = [results[(name, s)]["final_param_count"] for s in SEEDS]
    m = statistics.mean(vals)
    sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
    summary[name] = (m, sd, statistics.mean(params))
    print(
        f"  {name:>16s} {m:.4f} ± {sd:.4f}  (ppl ≈ {safe_perplexity(m):.2f})  "
        f"params={statistics.mean(params):,.0f}"
    )

# 자동 verdict
print("\n== Verdict ==")
small_loss = summary["small_baseline"][0]
grown_loss = summary["grown"][0]
large_loss = summary["large_baseline"][0]

# 1. all-finite
all_finite = all(math.isfinite(out["final_loss"]) for out in results.values())
verdict_1 = "PASS" if all_finite else "FAIL"
print(f"1. all-finite (grow stability): {all_finite}  [{verdict_1}]")

# 2. grown < small_baseline — grow 가 capacity 부족 모델보다 명확히 좋음
diff_grown_small = grown_loss - small_loss
verdict_2 = "PASS" if diff_grown_small < 0 else "FAIL"
print(f"2. grown < small_baseline: diff = {diff_grown_small:+.4f}  [{verdict_2}]")

# 3. grown ≤ large_baseline + 0.05 — grown 이 처음부터 큰 모델 근방까지 학습
diff_grown_large = grown_loss - large_loss
verdict_3 = "PASS" if diff_grown_large <= 0.05 else "FAIL"
print(f"3. grown ≤ large_baseline + 0.05: diff = {diff_grown_large:+.4f}  [{verdict_3}]")

# 4. grow 직후 function preservation 검증 — 학습 curve 의 step 750 근방에 spike 없음
# (정량 검증은 별도 unit test 가 atol=1e-5 로 보장; 여기서는 학습 curve smoothness 만 추정)
for seed in SEEDS:
    losses = results[("grown", seed)]["losses"]
    # step 750 직전 100step 평균 vs step 750 직후 1 step 비교 (drift 측정)
    before = sum(losses[max(0, GROW_AT_STEP - 100) : GROW_AT_STEP]) / 100
    after = losses[GROW_AT_STEP] if len(losses) > GROW_AT_STEP else losses[-1]
    spike = after - before
    print(
        f"   seed={seed}: grow 직전 100 step 평균 = {before:.4f}, grow 직후 1 step = {after:.4f}, spike = {spike:+.4f}"
    )

# grow 이벤트 정보
print("\n== Grow events ==")
for (name, seed), out in results.items():
    if out["grow_event"] is not None:
        ev = out["grow_event"]
        print(
            f"  {name} seed={seed}: step={ev['step']}, layers_grown={ev['n_layers_grown']}, "
            f"ffn {ev['old_ffn_dim']} → {ev['new_ffn_dim']}, n_new_groups={ev['n_new_groups']}"
        )

## 4. Loss curve 시각화 — grow step 표시

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = {
    "small_baseline": "tab:red",
    "grown": "tab:blue",
    "large_baseline": "tab:green",
}
window = 50

for mode in MODES:
    name = mode["name"]
    losses_per_seed = [results[(name, s)]["losses"] for s in SEEDS]
    smoothed = []
    for losses in losses_per_seed:
        smoothed.append(
            [
                sum(losses[max(0, i - window + 1) : i + 1]) / min(i + 1, window)
                for i in range(len(losses))
            ]
        )
    arr = torch.tensor(smoothed)
    mean = arr.mean(dim=0)
    std = arr.std(dim=0)
    steps = list(range(len(mean)))
    ax.plot(steps, mean, label=name, color=colors[name], linewidth=1.5)
    ax.fill_between(steps, mean - std, mean + std, color=colors[name], alpha=0.15)

# grow step 수직선
ax.axvline(GROW_AT_STEP, color="black", linestyle=":", alpha=0.5, label=f"grow @ {GROW_AT_STEP}")

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title(
    f"Phase 16b — Net2Net grow (ffn {FFN_SMALL} → {FFN_LARGE}): small / grown / large (mean ± σ over 2 seeds)"
)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase16b")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. Parameter count 비교

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
names = [m["name"] for m in MODES]
params = [summary[n][2] for n in names]
losses = [summary[n][0] for n in names]
stds = [summary[n][1] for n in names]

color_list = [colors[n] for n in names]
ax.errorbar(params, losses, yerr=stds, marker="o", capsize=4, linewidth=1.5, color="tab:gray")
for p, lo, n in zip(params, losses, names, strict=True):
    ax.annotate(n, (p, lo), xytext=(8, -8), textcoords="offset points", fontsize=10)

ax.set_xlabel("final parameter count")
ax.set_ylabel("final loss (last 100 mean ± σ)")
ax.set_title("Phase 16b — params vs loss: grown vs static baselines")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(out_dir / "params_vs_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'params_vs_loss.png'}")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

- grow 직후 function preservation 확인 (spike 없음)?
- grown 의 final loss 가 large_baseline 근방까지 회복?
- 16a (DST) 와 16b (grow) 의 의미 비교 — 16b 의 capacity 증가가 더 효과적?

**Phase 17 후보**:
- 16a + 16b 결합 — grow + shrink 동시 dynamic
- layer-wise 차등 (attention vs FFN 별 다른 grow / prune 정책)
- hidden_dim 까지 grow (downstream layer 모두 영향 — 더 invasive)